> ⚠️ **DO NOT run the code in this notebook directly!** 
> This script was used with the full DTU dataset downloaded.
>
> 💡 **If you want to run it:**
> 1. Download the dataset from [SGD Repository DTU Zenodo](https://doi.org/10.5281/zenodo.8202150).
> 2. Point the path in the code to your local directory: `./your_machine_path/sgd_data/square_runs`.

# Re-evaluating DTU Results with my metrics and classes 

The original work used TopFarm to compute AEP and constraints, but here we are using our own components to compare this version of SGD in WESL optimizer's AEP numbers against their SGD numbers.

So to ensure the AEP and all the metrics will be calculated equaly in this comparison, we are using the same WESL components and re-evaluating every optimization recorders they published.

### What the Script Does
I loaded the same .npy files they saved (turbine positions at each iteration), and for every case I computed:

- AEP using my AEPComp (with BastankhahGaussian wake model)

- Spacing constraints using my SpacingConstraintComp (min 2D = 160m)

- Boundary constraints using my BoundaryConstraintComp (square domain)

- RMS violation using my FinalRMSViol (aggregated violation)

I did this for both the initial design (iteration 0) and the final design (last iteration), so I can compare start and end points 

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from aep import AEPComp  
from constraints import SpacingConstraintComp, BoundaryConstraintComp
from aggregator import ConstraintAggregator
from penalty import FinalRMSViol

from py_wake.examples.data.lillgrund import LillgrundSite
from py_wake.examples.data.hornsrev1 import HornsrevV80
from py_wake.deficit_models.gaussian import BastankhahGaussian

site = LillgrundSite()
site.interp_method = 'linear'
wfm = BastankhahGaussian(site, HornsrevV80())

base_path = Path("/Users/brunoboer/Downloads/sgd_data/square_runs")#my local machine
output = Path(__file__).parent / "DTU_Recorders_Init_End_Full" #you can find the output files here: Optimizers_Bruno/DTU_Recorders_Init_End
output.mkdir(exist_ok=True)

row_configs = {
    10: (100, (10-1)*5*80),
    12: (144, (12-1)*5*80),
    15: (225, (15-1)*5*80),
    18: (324, (18-1)*5*80)
}

D = 80.0
min_spacing_m = 2 * D

def process_init_end(rows_str, n_wt, L, data_dir, case_type, seed, prefix='sgd'):
    data_path = data_dir / f"seed_{seed}"
    
    x_file = data_path / f'{prefix}_x.npy'
    y_file = data_path / f'{prefix}_y.npy'
    time_file = data_path / f'{prefix}_time.npy'
    
    if not all(f.exists() for f in [x_file, y_file, time_file]):
        return False
    
    x_pos = np.load(x_file)
    y_pos = np.load(y_file)
    times = np.load(time_file)
    
    T = x_pos.shape[0]
    iters = [0, T-1]  
    
    boundary_vertices = np.array([[0,0], [L,0], [L,L], [0,L]])
    
    aep_comp = AEPComp(wake_model=wfm, wt_x=np.zeros(n_wt), wt_y=np.zeros(n_wt)); aep_comp.setup()
    spacing_comp = SpacingConstraintComp(n_turbines=n_wt, min_spacing=min_spacing_m); spacing_comp.setup()
    boundary_comp = BoundaryConstraintComp(boundary_vertices=boundary_vertices, n_turbines=n_wt); boundary_comp.setup()
    agg_comp = ConstraintAggregator(n_turbines=n_wt); agg_comp.setup()
    rms_comp = FinalRMSViol(nconstraints=agg_comp.m_total); rms_comp.setup()
    
    rows = []
    time0 = times[0]
    
    for i, iter_idx in enumerate(iters):
        x_pos_i = x_pos[iter_idx]
        y_pos_i = y_pos[iter_idx]
        time_s = times[iter_idx] - time0
        
        inputs = {'x': x_pos_i, 'y': y_pos_i}; outputs = {'aep': np.zeros(1)}
        aep_comp.compute(inputs, outputs); aep = float(outputs['aep'])
        
        inputs_s = {'x': x_pos_i, 'y': y_pos_i}; outputs_s = {'spacing_cons': np.zeros(spacing_comp.m)}
        spacing_comp.compute(inputs_s, outputs_s); spacing_cons = outputs_s['spacing_cons']
        
        inputs_b = {'x': x_pos_i, 'y': y_pos_i}; outputs_b = {'boundary_cons': np.zeros(n_wt)}
        boundary_comp.compute(inputs_b, outputs_b); boundary_cons = outputs_b['boundary_cons']
        
        inputs_g = {'spacing_cons': spacing_cons, 'boundary_cons': boundary_cons}
        outputs_g = {'g_vector': np.zeros(agg_comp.m_total)}
        agg_comp.compute(inputs_g, outputs_g); g_vector = outputs_g['g_vector']
        
        inputs_r = {'g_vector': g_vector}; outputs_r = {'rms_viol': np.array(0.0)}
        rms_comp.compute(inputs_r, outputs_r); rms_viol = float(outputs_r['rms_viol'])
        
        rows.append({
            'eval': iter_idx+1, 'time_s': time_s, 'iter': iter_idx,
            'obj': -aep, 'aep': aep, 'rms_viol': rms_viol,
            'x': ';'.join(f"{xi:.3f}" for xi in x_pos_i),
            'y': ';'.join(f"{yi:.3f}" for yi in y_pos_i),
            'label': 'init' if iter_idx == 0 else 'end'
        })
    
    df = pd.DataFrame(rows)
    pasta_saida = output / case_type / f"rows_{rows_str}_T"
    pasta_saida.mkdir(parents=True, exist_ok=True)
    
    if prefix == 'sgd':
        csv_name = f'dtu_{n_wt}wt_T{T}_{prefix}_{case_type}_seed{seed}.csv'
    else:
        csv_name = f'dtu_{n_wt}wt_T{T}_{case_type}_seed{seed}.csv'
    
    df.to_csv(pasta_saida / csv_name, index=False)
    return True

print("DTU Init+End FULL")

print("\n📁 SGD 500/1000/2000...")
for rows_str, (n_wt, L) in row_configs.items():
    for T_target in [500, 1000, 2000]:
        sgd_dir = base_path / f"rows_{rows_str}_T_{T_target}"
        if sgd_dir.exists():
            print(f"  {sgd_dir.name}...")
            for seed in range(1, 21):
                process_init_end(rows_str, n_wt, L, sgd_dir, f"T{T_target}", seed, 'sgd')

print("\n📁 SLSQP det...")
for rows_str, (n_wt, L) in row_configs.items():
    det_dir = base_path / f"rows_{rows_str}_det"
    if det_dir.exists():
        print(f"  {det_dir.name}...")
        for seed in range(1, 21):
            process_init_end(rows_str, n_wt, L, det_dir, "slsqp", seed, 'det')

print(f"\n CSVs saved to {output}")

For each case, the csv file contains :

```
| Column   | What it is                                      |
| -------- | ----------------------------------------------- |
| eval     | Evaluation number (1 = initial, last = final)   |
| time_s   | Time elapsed in seconds                         |
| iter     | Iteration index (0 or T-1)                      |
| obj      | Objective value (-AEP)                          | 
| aep      | Annual Energy Production in MW                  |
| rms_viol | RMS constraint violation                        |
| x        | All turbine x coordinates (semicolon-separated) |
| y        | All turbine y coordinates (semicolon-separated) |
| label    | 'init' or 'end'                                 |
```

** obj is the -AEP because in my optimization, I enter this value negative. But here is just re-validation using the same components.


Then, the utput folder structure is organized in this way:
```
DTU_Recorders_Init_End/
    T500/
        rows_10_T/
            dtu_100wt_T500_sgd_T500_seed1.csv
            dtu_100wt_T500_sgd_T500_seed2.csv
            ...
        rows_12_T/
            ...
    T1000/
        ...
    T2000/
        ...
    slsqp/
        rows_10_T/
            dtu_100wt_Tslsqp_seed1.csv
            dtu_100wt_Tslsqp_seed2.csv
            ...
```